# 10. LLM을 활용한 텍스트마이닝 기법


## 1. 규칙·통계 기반 분석에서 LLM API로

이전에는 텍스트를 수집하고, 정제하고, 토큰화하는 방법을 다뤘습니다. 정규표현식으로 이메일·URL·전화번호를 찾고, 불용어를 제거하고, 형태소 분석으로 명사를 뽑는 작업은 규칙이 분명할 때 잘 작동합니다.

또한 텍스트를 숫자로 바꾸는 방법도 살펴봤습니다. BoW, TF-IDF, N-gram, Word2Vec, 로지스틱 회귀를 사용하면 단어 빈도와 패턴을 바탕으로 검색, 키워드 확인, 감성 분류 같은 기본 분석을 만들 수 있습니다.

하지만 모든 텍스트 문제가 규칙이나 통계만으로 깔끔하게 해결되지는 않습니다. 다음과 같은 작업은 결국 사람이 문맥을 읽고 판단해야 하는 경우가 많습니다.

| 상황 | 규칙·통계 기반 접근의 한계 | LLM API로 할 수 있는 일 |
|---|---|---|
| 긴 기사나 회의록 요약 | 어떤 문장이 중요한지 단어 빈도만으로 판단하기 어려움 | 핵심 주장, 배경, 결론을 문장으로 요약 |
| 리뷰의 복합 감정 판단 | `좋다`, `별로다` 같은 단어만으로 긍정/부정을 나누기 어려움 | 긍정·부정·중립과 판단 근거를 함께 출력 |
| 고객 문의 분류 | 새 표현, 은어, 복합 문의가 나오면 규칙을 계속 추가해야 함 | 정해진 카테고리 중 하나로 분류하고 이유 설명 |
| 키워드와 이슈 추출 | 자주 나온 단어가 항상 중요한 이슈는 아님 | 문맥상 중요한 키워드와 이유를 함께 추출 |
| 사람이 검토해야 하는 문서 정리 | 최종 판단은 사람이 해야 하지만 모든 문서를 직접 읽기엔 시간이 오래 걸림 | 1차 요약, 위험 신호, 검토 우선순위 생성 |

LLM API는 사람의 검토를 완전히 없애는 도구라기보다, 사람이 읽고 판단해야 하는 텍스트를 먼저 정리해 주는 도구에 가깝습니다. 이 장에서는 LLM을 사용해 요약, 키워드 추출, 감정 분류, 카테고리 분류를 자동화하고, 임베딩 API로 문서 유사도 검색을 구성합니다.


### 1.1 LDA로 주제 모델링 이해하기

LDA(Latent Dirichlet Allocation)는 여러 문서 안에 반복해서 나타나는 **잠재 주제(topic)** 를 찾는 토픽 모델링 방법입니다. 여기서 잠재 주제란 사람이 미리 붙여 둔 라벨이 아니라, 단어들이 함께 등장하는 패턴을 보고 모델이 찾아낸 숨은 묶음입니다.

머신러닝 관점에서 LDA는 **비지도학습(unsupervised learning)** 입니다. `이 문서는 배송 문제`, `이 문서는 품질 문제`처럼 정답 라벨을 미리 달아두지 않아도, 문서 안의 단어 빈도와 함께 등장하는 패턴만 보고 주제를 찾아냅니다.

예를 들어 쇼핑몰 리뷰를 모아 보면 어떤 문서에는 `배송, 택배, 지연, 도착` 같은 단어가 자주 함께 나오고, 다른 문서에는 `품질, 소음, 디자인, 성능` 같은 단어가 자주 함께 나올 수 있습니다. LDA는 이런 반복 패턴을 이용해 문서 묶음 안에 어떤 주제가 있는지 추정합니다.

LDA는 문서를 다음처럼 바라봅니다.

- **문서(document)** 는 하나의 주제로만 이루어진 것이 아니라 여러 주제가 섞인 결과입니다.
- **주제(topic)** 는 특정 단어들이 높은 확률로 등장하는 단어 묶음입니다.
- **단어(word)** 는 문서 안에서 관찰되는 실제 데이터입니다.

### LDA 학습 단계

학습 단계에서는 정답 라벨이 없는 문서 묶음만 가지고 시작합니다. 먼저 문서를 단어 빈도 행렬로 바꾸고, 찾고 싶은 주제 개수를 정한 뒤, 각 단어가 어떤 주제에 속한다고 보는 것이 자연스러운지 반복해서 조정합니다.

이때 모델은 두 가지를 함께 맞춰 갑니다. 한 문서 안에서는 비슷한 주제의 단어들이 자주 함께 나와야 하고, 한 주제 안에서는 서로 관련 있는 단어들이 높은 비중을 가져야 합니다. 이런 반복 학습이 끝나면 문서별 주제 비율과 주제별 단어 비중이 만들어집니다.

<img src="image/lda_training_flow.svg" width="820">

### LDA 학습 이후 결과 해석

학습이 끝난 뒤에는 모델이 만든 결과를 사람이 해석합니다. `Topic 0`, `Topic 1` 같은 번호 자체에는 의미가 없고, 각 주제에서 높은 비중을 가진 대표 단어를 보고 `배송 이슈`, `제품 품질`, `가격/혜택`처럼 이름을 붙입니다.

기존 그림은 이 학습 이후 단계, 즉 LDA 결과를 해석하고 문서별로 어떤 주제가 강한지 확인하는 흐름을 보여줍니다.

<img src="image/lda_topic_model_flow.svg" width="820">

즉 LDA의 핵심은 두 가지를 동시에 추정하는 것입니다.

| 추정 대상 | 의미 | 예시 |
|---|---|---|
| 문서별 주제 비율 | 한 문서 안에 어떤 주제가 얼마나 섞여 있는가 | 리뷰 A = 배송 70%, 가격 20%, 품질 10% |
| 주제별 단어 비중 | 한 주제를 설명하는 대표 단어가 무엇인가 | 배송 주제 = 배송, 택배, 지연, 도착 |

### LDA 분석 흐름

1. 여러 문서를 준비합니다.
2. 텍스트를 정제하고 단어 단위로 나눕니다.
3. `CountVectorizer`처럼 단어 등장 횟수 행렬을 만듭니다.
4. 몇 개의 주제를 찾을지 `n_components`로 정합니다.
5. LDA 모델을 학습해 문서별 주제 비율과 주제별 대표 단어를 얻습니다.
6. 사람이 대표 단어를 보고 `배송 이슈`, `제품 품질`, `가격/혜택`처럼 주제 이름을 해석합니다.

주의할 점은 LDA가 `배송 이슈` 같은 이름을 직접 붙여주지는 않는다는 것입니다. 모델은 `Topic 0`, `Topic 1`처럼 번호와 대표 단어를 보여주고, 그 의미를 읽고 이름 붙이는 일은 분석자가 해야 합니다.

아래 실습에서는 더미 리뷰를 직접 만든 뒤, LDA가 어떤 단어 묶음을 주제로 찾아내는지 단계별로 확인합니다.




In [ ]:
import pandas as pd  # LDA 결과를 표로 보기 위해 사용합니다.
from sklearn.decomposition import LatentDirichletAllocation  # LDA 토픽 모델입니다.
from sklearn.feature_extraction.text import CountVectorizer  # 문서를 단어 빈도 행렬로 바꾸는 도구입니다.

# 외부 파일 없이 실습하기 위해 만든 더미 리뷰입니다.
dummy_reviews = [
    "배송 빠름 배송 알림 도착 택배 포장",
    "택배 지연 배송 느림 도착 지연 고객센터",
    "배송 추적 알림 빠름 도착 일정 택배",
    "주말 배송 지연 택배 도착 늦음",
    "성능 만족 품질 튼튼 소음 적음 디자인",
    "제품 품질 불량 전원 고장 교환 필요",
    "소음 작음 성능 좋음 흡입력 만족",
    "디자인 깔끔 내구성 좋음 품질 만족",
    "가격 할인 쿠폰 이벤트 혜택 저렴",
    "가격 비쌈 할인 부족 쿠폰 적용",
    "멤버십 혜택 적립 할인 가격 쿠폰",
    "프로모션 쿠폰 가격 저렴 재구매",
]


In [ ]:
topic_df = pd.DataFrame({
    "document_id": range(1, len(dummy_reviews) + 1),
    "text": dummy_reviews,
})

topic_df


In [ ]:
vectorizer = CountVectorizer()  # 문서를 단어 빈도 행렬로 바꾸는 도구입니다.

document_term_matrix = vectorizer.fit_transform(topic_df["text"])  # 문서-단어 행렬을 만듭니다.
feature_names = vectorizer.get_feature_names_out()  # CountVectorizer가 만든 단어 목록입니다.

print("문서-단어 행렬 크기:", document_term_matrix.shape)
print("단어 예시:", feature_names[:10])


In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=3,  # 찾고 싶은 주제 개수입니다. 배송, 품질, 가격 정도를 기대해 볼 수 있습니다.
    random_state=42,  # 실행할 때마다 비슷한 결과가 나오도록 고정합니다.
)
document_topic_matrix = lda_model.fit_transform(document_term_matrix)  # 문서별 주제 비율을 학습합니다.

print("문서-주제 행렬 크기:", document_topic_matrix.shape)


In [ ]:
topic_rows = []
for topic_index, topic_weights in enumerate(lda_model.components_):
    top_indices = topic_weights.argsort()[-6:][::-1]  # 각 주제에서 가중치가 큰 단어 6개를 고릅니다.
    topic_rows.append({
        "topic": f"Topic {topic_index}",
        "top_words": ", ".join(feature_names[top_indices]),
    })

topic_words_df = pd.DataFrame(topic_rows)
topic_words_df  # 주제별 대표 단어를 보고 사람이 주제 이름을 붙입니다.


In [ ]:
topic_result_df = topic_df.copy()
topic_result_df["dominant_topic"] = document_topic_matrix.argmax(axis=1)  # 가장 비중이 큰 주제 번호입니다.
topic_result_df["topic_confidence"] = document_topic_matrix.max(axis=1).round(3)  # 해당 주제 비중입니다.

topic_result_df  # 문서마다 어떤 주제가 가장 강한지 확인합니다.


LDA 결과의 `Topic 0`, `Topic 1` 같은 번호는 자동으로 붙은 이름일 뿐입니다. 대표 단어를 보고 사람이 `배송 이슈`, `제품 품질`, `가격/혜택`처럼 해석 이름을 붙여야 합니다.  
주제 이름을 붙인 뒤에는 문서별 주제 비율을 확인해 어떤 리뷰가 어떤 이슈에 가까운지 살펴볼 수 있습니다. 이렇게 하면 많은 문서를 일일이 읽기 전에 전체 이슈 구조를 빠르게 파악할 수 있습니다.


## 2. GPT API 이해하기

### API란?

API(Application Programming Interface)는 한 프로그램이 다른 프로그램의 기능을 사용할 수 있게 해주는 약속입니다. 웹사이트에서 날씨를 불러오거나, 결제 시스템과 연결하거나, 지도 정보를 가져오는 것도 API를 통해 이루어집니다.

OpenAI의 GPT API는 내 컴퓨터에 GPT 모델을 직접 설치하지 않고, 코드에서 OpenAI 서버로 요청을 보내 답변을 받는 방식입니다.

```text
내 노트북 코드 -> OpenAI API 요청 -> GPT 모델 처리 -> 응답 반환 -> 파이썬에서 결과 활용
```

### 왜 API로 LLM을 사용할까?

LLM을 직접 내 컴퓨터나 회사 서버에서 돌리려면 모델 파일, 추론 서버, GPU 메모리, 운영 환경을 모두 준비해야 합니다. 특히 큰 언어 모델은 고가의 GPU가 필요하고, 여러 사람이 동시에 쓰거나 긴 문서를 처리하려면 장비와 운영 비용이 빠르게 커집니다.

반면 API를 사용하면 모델을 직접 설치하고 운영하지 않아도 됩니다. 필요한 텍스트를 요청으로 보내고, 사용한 만큼만 비용을 내고, 결과를 파이썬 코드에서 바로 데이터로 받을 수 있습니다. 수업이나 실무 자동화 관점에서는 **비싼 모델 인프라를 직접 소유하는 대신, 필요한 순간에 LLM 기능을 호출하는 방식**이라고 볼 수 있습니다.

이전까지의 방식으로도 워드클라우드, 단어 빈도, TF-IDF, 간단한 분류 모델은 자동으로 만들 수 있었습니다. 하지만 여러 뉴스 기사를 읽고 오늘의 핵심 흐름을 요약하거나, 회의록에서 결론과 액션 아이템을 뽑거나, 복합적인 고객 리뷰의 의도를 판단하는 일은 단어 빈도만으로 해결하기 어렵습니다.

그래서 이번 장에서는 규칙 기반 코드가 잘하는 일과 LLM이 잘하는 일을 나눠서 생각합니다. 파이썬은 데이터를 수집하고 정리하고 저장하는 반복 작업을 맡고, LLM API는 사람이 읽고 판단해야 했던 요약·분류·키워드 추출 같은 작업을 대신 처리합니다. 말하자면 **기존 방식으로 해결하기 어려웠던 문맥 판단 작업을 LLM에게 외주 주는 구조**입니다.

ChatGPT 웹/앱과 GPT API는 사용 방식이 다릅니다.

| 구분 | ChatGPT 웹/앱 | GPT API |
|---|---|---|
| 사용 방식 | 사람이 화면에서 직접 대화 | 파이썬 코드가 모델을 호출 |
| 주요 목적 | 개인 업무 보조, 대화 | 서비스 개발, 자동화, 데이터 처리 |
| 과금 기준 | ChatGPT 구독 요금제 | API 사용량 기반 별도 과금 |
| 결과 활용 | 화면에서 읽고 복사 | 데이터프레임, 파일, 앱, 서비스에 연결 |

GPT API를 사용하면 모델의 답변을 파이썬 코드 안에서 데이터로 받아 요약, 분류, 키워드 추출 같은 작업에 연결할 수 있습니다.


### API Key와 `.env` 파일

GPT API를 사용하려면 API Key가 필요합니다. API Key는 OpenAI가 `누가 요청을 보냈는지`, `얼마나 사용했는지`, `어느 계정에 과금해야 하는지`를 확인하는 비밀 인증 값입니다.

API Key는 비밀번호처럼 다뤄야 합니다.

- 노트북 코드 안에 직접 적지 않습니다.
- GitHub, 블로그, 캡처 화면에 노출하지 않습니다.
- 다른 사람에게 공유하지 않습니다.
- 유출이 의심되면 즉시 삭제하고 새로 발급합니다.

실습 폴더 루트에 `.env` 파일을 만들고 아래처럼 저장합니다.

```bash
OPENAI_API_KEY=sk-...
```

아래 셀의 `load_dotenv()`는 `.env` 파일에 저장된 API Key를 현재 파이썬 실행 환경으로 불러옵니다.


### GPT API 모델과 요금 이해하기

OpenAI API에서는 `model` 값으로 어떤 모델을 사용할지 정합니다. 예를 들어 `ChatOpenAI(model="gpt-4o-mini")`라고 쓰면 LangChain이 OpenAI의 `gpt-4o-mini` 모델에 요청을 보냅니다.

모델을 고를 때는 아래 기준을 함께 봅니다.

- 성능: 복잡한 추론, 긴 글 이해, 코드 작성이 필요한가?
- 비용: 같은 작업을 많이 반복할 예정인가?
- 속도: 답변이 빠르게 돌아와야 하는가?
- 입력 형태: 텍스트만 쓰는가, 이미지도 넣는가?
- 출력 형태: 자유 문장인가, JSON 같은 정해진 형식인가?

아래 가격은 2026-05-27 기준 USD 가격입니다. 최신 가격은 공식 Pricing 문서에서 확인할 수 있습니다.

| 모델 | 설명 | 입력 가격 / 1M tokens | 출력 가격 / 1M tokens | 추천 상황 |
|---|---|---:|---:|---|
| `gpt-5.5` | 최신 플래그십 모델. 복잡한 추론과 코딩에 강하지만 비용이 높습니다. | $5.00 | $30.00 | 어려운 분석, 고난도 추론, 중요한 코드 작업 |
| `gpt-5.4` | 고성능 작업용 모델. `gpt-5.5`보다 저렴한 상위 모델입니다. | $2.50 | $15.00 | 품질이 중요한 문서 분석, 전문 업무 자동화 |
| `gpt-5.4-mini` | 성능과 비용의 균형을 맞춘 소형 모델입니다. | $0.75 | $4.50 | 대량 처리, 업무형 챗봇, 실무 자동화 |
| `gpt-5.4-nano` | 단순하고 반복적인 작업에 적합한 저비용 모델입니다. | $0.20 | $1.25 | 분류, 태깅, 정보 추출, 빠른 대량 처리 |
| `gpt-4o-mini` | 빠르고 저렴한 소형 모델입니다. | $0.15 | $0.60 | 입문 실습, 요약, 키워드 추출, 감정 분류 |
| `text-embedding-3-small` | 텍스트를 숫자 벡터로 바꾸는 임베딩 모델입니다. | $0.02 | 해당 없음 | 유사도 검색, 추천, RAG 검색 단계 |

### 토큰이란?

토큰은 모델이 글을 읽고 쓸 때 사용하는 작은 단위입니다. `1글자 = 1토큰`처럼 고정된 규칙은 아니고, 토크나이저가 글자·단어·자주 나오는 조각을 섞어서 나눕니다.

간단히 감을 잡으면 다음과 같습니다.

| 예시 | 토큰 감각 |
|---|---|
| 영어 `hello` | 짧은 단어 하나가 1토큰 정도로 처리될 수 있음 |
| 영어 `hello world` | 보통 `hello`, `world`처럼 단어 단위에 가깝게 나뉨 |
| 한글 `안녕하세요` | 글자별로 나뉘거나 자주 쓰는 조각이 묶여 몇 토큰이 됨 |
| 한글 `배송이 빠릅니다` | 단어, 조사, 띄어쓰기 주변 조각이 섞여 여러 토큰이 됨 |

따라서 영어는 보통 **1토큰이 여러 글자나 짧은 단어 하나**에 가깝고, 한글은 **1글자가 1토큰 안팎**으로 잡히는 경우가 많습니다. 정확한 토큰 수는 모델의 토크나이저로 직접 세어 봐야 합니다.

수업에서는 vocab 목록을 직접 열어 보면 토큰이 어떤 조각으로 구성되는지 감을 잡기 쉽습니다. 아래 GitHub Raw 링크는 `o200k_base` vocab을 사람이 읽을 수 있게 디코딩해 둔 목록입니다.

- `o200k_base` decoded vocab 목록: https://raw.githubusercontent.com/kaisugi/gpt4_vocab_list/main/o200k_base_vocab_list.txt

링크를 열면 앞부분은 다음처럼 보입니다. 왼쪽 번호는 설명을 위해 붙인 토큰 ID 예시이고, 오른쪽 값이 실제 토큰 조각입니다.

```text
0: '!'
1: '"'
2: '#'
3: '$'
4: '%'
...
```

공식 참고 링크:

- Models: https://developers.openai.com/api/docs/models
- Pricing: https://developers.openai.com/api/docs/pricing





In [ ]:
import tiktoken  # OpenAI 모델의 토큰 수를 세는 라이브러리입니다.

encoding = tiktoken.get_encoding("o200k_base")  # 최신 GPT 계열에서 쓰는 토크나이저 계열입니다.

sample_texts = [
    "hello",
    "hello world",
    "안녕하세요",
    "배송이 빠릅니다",
]

for text in sample_texts:
    token_ids = encoding.encode(text)
    print(f"{text!r} -> {len(token_ids)} tokens -> {token_ids}")



In [ ]:
from dotenv import load_dotenv  # .env 파일에 저장된 환경변수를 불러오는 함수입니다.

load_dotenv()  # 현재 폴더의 .env 파일을 읽어 OpenAI API 키를 사용할 수 있게 합니다.


### LangChain 기본 구성

LangChain에서는 보통 다음 세 가지를 연결해 사용합니다.

- `ChatPromptTemplate`: 모델에게 전달할 지시문과 입력 형식
- `ChatOpenAI`: OpenAI 채팅 모델 호출
- `StrOutputParser` 또는 `JsonOutputParser`: 응답을 문자열이나 JSON 형태로 정리

아래 코드는 이번 차시에서 반복해서 사용할 기본 객체를 준비합니다.


### 프롬프트 템플릿이란?

`ChatPromptTemplate.from_template()`은 모델에게 보낼 프롬프트를 하나의 문자열로 작성하는 방식입니다. 처음 배울 때는 역할별 메시지를 나누는 방식보다 이 형태가 더 직관적입니다.

프롬프트 안에는 다음 내용을 한꺼번에 적을 수 있습니다.

- 모델에게 맡길 역할
- 수행할 작업
- 원하는 출력 형식
- 분석할 입력 데이터 자리

입력 데이터가 들어갈 자리는 `{text}`, `{question}`처럼 중괄호로 표시합니다.

```python
prompt = ChatPromptTemplate.from_template("""
너는 텍스트마이닝 분석가야.

다음 텍스트를 3문장 이내로 요약해줘.

텍스트:
{text}
""")
```

이후 `chain.invoke({"text": sample_text})`처럼 값을 넣으면 `{text}` 자리에 실제 데이터가 들어갑니다. 이 장에서는 프롬프트를 쉽게 읽고 수정할 수 있도록 `from_template()` 중심으로 실습합니다.




In [ ]:
import json  # JSON 형태의 문자열과 파이썬 객체를 다룰 때 사용합니다.
import numpy as np  # 임베딩 벡터의 유사도 계산에 사용할 수치 연산 라이브러리입니다.
import pandas as pd  # 표 형태의 결과를 보기 좋게 정리하는 라이브러리입니다.

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 LangChain에서 사용합니다.
from langchain_core.prompts import ChatPromptTemplate  # 프롬프트 템플릿을 구성합니다.
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser  # 모델 응답을 문자열 또는 JSON으로 정리합니다.

chat_model_name = "gpt-4o-mini"  # 실습 기본 모델입니다. 필요하면 "gpt-5.4-nano" 등으로 바꿔 비교할 수 있습니다.
llm = ChatOpenAI(model=chat_model_name, temperature=0)  # temperature=0은 답변을 비교적 일관되게 만듭니다.



## 3. 텍스트 요약

요약은 긴 리뷰, 뉴스, 회의록, 상담 기록에서 핵심 내용을 빠르게 파악할 때 자주 사용됩니다.  
요약 프롬프트에서는 **무엇을 남길지**, **얼마나 짧게 만들지**, **어떤 형식으로 출력할지**를 명확히 정합니다.


In [ ]:
sample_text = """
최근 한 달간 온라인 쇼핑몰의 생활가전 카테고리 리뷰 1,200건을 살펴보았다.
고객들은 대체로 제품 성능과 가격 대비 만족도에는 긍정적인 반응을 보였다.
특히 공기청정기와 무선청소기 상품에서는 흡입력, 소음 수준, 디자인에 대한 칭찬이 많았다.
배송 속도도 전반적으로 빠르다는 의견이 많았고, 지정일 배송을 편리하게 느낀 고객도 있었다.
다만 포장 상태에 대한 불만은 여러 상품군에서 반복적으로 나타났다.
일부 고객은 외부 박스가 찌그러져 도착했거나 완충재가 부족해 제품 파손을 걱정했다고 작성했다.
교환과 환불 절차에 대해서는 안내가 복잡하고 처리 상태를 확인하기 어렵다는 의견이 많았다.
고객센터 응답 속도도 주요 불만 요인으로 언급되었으며, 특히 주말 문의의 답변 지연이 자주 등장했다.
반면 상담원이 문제를 빠르게 해결해준 사례에서는 브랜드 신뢰도가 높아졌다는 평가도 있었다.
리뷰 작성자들은 제품 자체에는 만족하지만 구매 이후 문제가 생겼을 때의 지원 경험이 아쉽다고 정리했다.
재구매 의사가 있는 고객들도 포장 개선, 교환 절차 단순화, 배송 상태 알림 강화를 요구했다.
종합하면 상품 경쟁력은 유지되고 있으나, 물류와 사후 지원 경험을 개선하면 고객 만족도를 더 높일 수 있다.
""".strip()  # 요약과 키워드 추출에 사용할 긴 예시 텍스트입니다.

print(sample_text[:500])


In [ ]:
summary_prompt = ChatPromptTemplate.from_template("""  # 하나의 문자열로 프롬프트를 구성합니다.
너는 텍스트마이닝 분석가야.

다음 텍스트를 3문장 이내로 요약해줘.

텍스트:
{text}
""")

summary_chain = summary_prompt | llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 출력 순서로 체인을 만듭니다.



In [ ]:
summary = summary_chain.invoke({"text": sample_text})  # {text} 자리에 sample_text를 넣어 체인을 실행합니다.
print(summary)  # 요약 결과를 출력합니다.


### 실무 예시: 오늘 경제 뉴스 요약

`naver_economy_news.csv`에 저장된 뉴스 제목과 본문을 LLM에 전달해 오늘 경제 뉴스의 흐름을 요약해 봅니다.  
기사 전체 본문을 모두 넣으면 입력이 길어질 수 있으므로, 아래 예시는 기사 수와 본문 길이를 제한해 실습 비용과 실행 시간을 조절합니다.


In [ ]:
news_df = pd.read_csv("naver_economy_news.csv")  # 오늘 수집한 네이버 경제 뉴스 CSV를 읽습니다.
news_df = news_df.dropna(subset=["title"]).copy()  # 제목이 없는 행은 요약 대상에서 제외합니다.
news_df["body"] = news_df["body"].fillna("")  # 본문이 비어 있는 경우 빈 문자열로 처리합니다.

news_df[["title", "press", "time"]].head()


In [ ]:
article_limit = 12  # 요약에 사용할 기사 수입니다. 전체를 쓰고 싶으면 값을 늘려 보세요.
body_char_limit = 700  # 기사 1건당 본문에서 사용할 글자 수입니다.

news_sample = news_df.head(article_limit).copy()  # 입력 길이를 조절하기 위해 일부 기사만 사용합니다.
print("요약에 사용할 기사 수:", len(news_sample))


In [ ]:
article_blocks = []  # LLM에 전달할 기사 묶음을 저장합니다.
for idx, row in news_sample.iterrows():
    article_blocks.append(
        f"[{idx + 1}] 제목: {row['title']}\n"
        f"언론사: {row.get('press', '')}\n"
        f"기자: {row.get('journalist', '')}\n"
        f"시간: {row.get('time', '')}\n"
        f"본문 일부: {row['body'][:body_char_limit]}"
    )

news_text = "\n\n".join(article_blocks)  # 여러 기사 내용을 하나의 입력 텍스트로 합칩니다.
print(news_text[:1200])  # 너무 길면 앞부분만 확인합니다.


In [ ]:
news_summary_prompt = ChatPromptTemplate.from_template("""  # 오늘 뉴스 요약용 프롬프트를 구성합니다.
너는 한국 경제 뉴스를 정리하는 데이터 저널리스트야.

다음은 오늘 수집한 경제 뉴스 제목과 본문 일부야.
중복되는 내용은 묶고, 오늘 뉴스의 핵심 흐름을 한국어로 요약해줘.

출력 형식:
1. 한줄 요약: 오늘 경제 뉴스의 가장 큰 흐름 1문장
2. 핵심 이슈 3가지: 각 이슈를 bullet로 정리
3. 주목할 기업/기관: 기사에서 많이 언급되거나 의미 있는 기업·기관
4. 수업 토론 질문: 뉴스 데이터를 더 분석하기 위한 질문 2개

뉴스 데이터:
{news_text}
""")

news_summary_chain = news_summary_prompt | llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 출력 체인입니다.



In [ ]:
today_news_summary = news_summary_chain.invoke({"news_text": news_text})  # 오늘 뉴스 요약을 생성합니다.
print(today_news_summary)  # 생성된 오늘 뉴스 요약을 출력합니다.


### 문제 1. 요약 관점 바꾸기

위 요약 프롬프트를 수정해 다음 형식으로 출력해 보세요.

- 핵심 긍정 의견
- 핵심 부정 의견
- 개선 제안

<details>
<summary>정답 보기</summary>

```python
news_summary_prompt = ChatPromptTemplate.from_template("""
너는 한국 경제 뉴스를 정리하는 데이터 저널리스트야.

다음은 오늘 수집한 경제 뉴스 제목과 본문 일부야.
긍정 의견, 부정 의견, 개선 제안 관점으로 나눠 한국어로 요약해줘.

출력 형식:
1. 핵심 긍정 의견: 오늘 뉴스에서 긍정적으로 볼 수 있는 흐름 2~3개
2. 핵심 부정 의견: 우려되거나 부정적으로 볼 수 있는 흐름 2~3개
3. 개선 제안: 기업, 정책 담당자, 소비자 관점에서 생각해 볼 제안 2~3개

뉴스 데이터:
{news_text}
""")

news_summary_chain = news_summary_prompt | llm | StrOutputParser()
today_news_summary = news_summary_chain.invoke({"news_text": news_text})
print(today_news_summary)
```
</details>



## 4. 키워드 추출

키워드 추출은 문서에서 중요한 단어와 짧은 구를 뽑아 이슈를 요약하는 작업입니다.  
LLM을 사용할 때는 결과를 표로 다루기 쉽도록 JSON 형식으로 받는 것이 편합니다.


In [ ]:
keyword_parser = JsonOutputParser()  # 모델 응답을 파이썬 dict/list로 바꿔주는 JSON 파서입니다.

keyword_prompt = ChatPromptTemplate.from_template("""  # 키워드 추출용 프롬프트를 구성합니다.
너는 한국어 텍스트마이닝 분석가야.

다음 텍스트에서 키워드 5개를 JSON으로 추출해줘.
반드시 아래 형식의 JSON만 출력해줘.

출력 형식:
{{"keywords": [{{"keyword": "...", "reason": "..."}}]}}

텍스트:
{text}
""")

keyword_chain = keyword_prompt | llm | keyword_parser  # 프롬프트 -> 모델 -> JSON 파서 순서로 연결합니다.
keyword_result = keyword_chain.invoke({"text": sample_text})  # 예시 텍스트에서 키워드를 추출합니다.
keyword_result  # 추출된 JSON 결과를 확인합니다.



In [ ]:
keyword_df = pd.DataFrame(keyword_result["keywords"])  # keywords 리스트를 표 형태로 변환합니다.
keyword_df  # 키워드와 추출 이유를 데이터프레임으로 확인합니다.


## 5. 감정 분류

감정 분류는 리뷰나 댓글을 긍정, 부정, 중립으로 나누는 작업입니다.  
LLM은 문장의 표현을 읽고 분류할 수 있지만, 결과 기준을 명확히 적어야 일관성이 좋아집니다.


In [ ]:
review_df = pd.DataFrame({  # 감정 분류에 사용할 짧은 리뷰 예시를 만듭니다.
    "text": [
        "배송이 정말 빨라서 만족합니다.",
        "제품은 괜찮지만 포장이 너무 허술했어요.",
        "교환 신청을 했는데 답변이 너무 늦습니다.",
        "가격 대비 품질이 좋아서 재구매하고 싶어요.",
        "아직 사용 전이라 평가는 어렵습니다.",
    ]
})
review_df  # 분류 대상 리뷰 목록을 확인합니다.


In [ ]:
sentiment_prompt = ChatPromptTemplate.from_template("""  # 감정 분류용 프롬프트를 구성합니다.
너는 고객 리뷰를 분석하는 분류기야.

다음 리뷰의 감정을 positive, negative, neutral 중 하나로 JSON 분류해줘.
반드시 아래 형식의 JSON만 출력해줘.

출력 형식:
{{"sentiment": "...", "reason": "..."}}

리뷰:
{text}
""")

sentiment_chain = sentiment_prompt | llm | JsonOutputParser()  # 프롬프트, 모델, JSON 파서를 하나의 체인으로 연결합니다.

sentiment_rows = []  # 리뷰별 분류 결과를 저장할 리스트입니다.
for text in review_df["text"]:  # 리뷰를 한 건씩 꺼내 모델에 전달합니다.
    sentiment_rows.append(sentiment_chain.invoke({"text": text}))  # 각 리뷰의 감정과 근거를 저장합니다.

sentiment_df = pd.concat([review_df, pd.DataFrame(sentiment_rows)], axis=1)  # 원본 리뷰와 분류 결과를 옆으로 붙입니다.
sentiment_df  # 최종 감정 분류 결과를 확인합니다.



## 6. 리뷰 평점화와 종합 보고서 작성

이번에는 수업용으로 만든 가상 상품 리뷰 CSV를 읽고, LLM이 각 리뷰를 1~5점으로 평가한 뒤 평균 평점과 종합 리뷰 보고서를 작성하는 예시를 봅니다.

이 작업은 광고 카피처럼 한두 문장을 만드는 일보다 API 활용에 더 잘 맞습니다. 여러 개의 텍스트를 반복 처리하고, 결과를 표로 저장하고, 마지막에 요약 보고서까지 자동으로 만들 수 있기 때문입니다.

주의할 점은 이 데이터가 실제 고객 리뷰가 아니라 **분석 실습용 가상 리뷰**라는 점입니다. 실제 서비스에서는 가짜 리뷰를 외부에 게시하거나 고객 반응처럼 속이면 안 됩니다. 여기서는 리뷰 분석 업무를 연습하기 위한 샘플 데이터로만 사용합니다.


In [ ]:
review_df = pd.read_csv("data/synthetic_product_reviews.csv")  # 수업용 가상 리뷰 데이터를 읽습니다.
display(review_df)  # 아직 llm_rating, rating_reason 컬럼은 비어 있습니다.


In [ ]:
review_llm = ChatOpenAI(model="gpt-5-nano")  # 여러 리뷰를 반복 처리하기 위해 비용이 낮은 모델을 사용합니다.
review_rating_parser = JsonOutputParser()  # 평점 결과를 JSON 형태로 받기 위한 파서입니다.


In [ ]:
review_rating_prompt = ChatPromptTemplate.from_template("""
너는 이커머스 리뷰 분석가입니다.
리뷰 내용을 읽고 고객 만족도를 1~5점 정수로 평가하세요.
1점은 매우 불만족, 3점은 보통 또는 장단점 혼재, 5점은 매우 만족을 의미합니다.
반드시 JSON만 출력하세요.

상품명: {product_name}
고객군: {customer_segment}
리뷰: {review_text}

출력 형식:
{{"rating": 4, "reason": "평점 근거 한 문장", "key_issue": "핵심 이슈 한 단어 또는 짧은 구"}}
""")

review_rating_chain = review_rating_prompt | review_llm | review_rating_parser  # 프롬프트 -> 모델 -> JSON 파서 체인입니다.



In [ ]:
rating_rows = []
for _, row in review_df.iterrows():  # 리뷰를 한 건씩 LLM에 전달합니다.
    result = review_rating_chain.invoke({
        "product_name": row["product_name"],
        "customer_segment": row["customer_segment"],
        "review_text": row["review_text"],
    })
    rating_rows.append(result)

rating_rows[:2]  # 생성 결과 일부를 먼저 확인합니다.


In [ ]:
rating_df = pd.DataFrame(rating_rows).rename(columns={"rating": "llm_rating", "reason": "rating_reason"})  # LLM 결과 컬럼명을 정리합니다.
review_scored_df = pd.concat([review_df.drop(columns=["llm_rating", "rating_reason"], errors="ignore"), rating_df], axis=1)
review_scored_df["llm_rating"] = pd.to_numeric(review_scored_df["llm_rating"], errors="coerce")  # 평균 계산을 위해 숫자형으로 변환합니다.

average_rating = review_scored_df["llm_rating"].mean().round(2)  # LLM이 평가한 평균 평점입니다.
print("LLM 평가 평균 평점:", average_rating)

review_scored_df[["review_id", "customer_segment", "review_text", "llm_rating", "rating_reason", "key_issue"]]


In [ ]:
review_scored_df.to_csv("data/synthetic_product_reviews_scored.csv", index=False, encoding="utf-8-sig")  # 평점이 추가된 결과를 저장합니다.
print("저장 완료: data/synthetic_product_reviews_scored.csv")


In [ ]:
rated_reviews_text = review_scored_df[["customer_segment", "review_text", "llm_rating", "rating_reason", "key_issue"]].to_string(index=False)
print(rated_reviews_text[:1200])


In [ ]:
review_report_prompt = ChatPromptTemplate.from_template("""
너는 마케팅 리서치 보고서를 작성하는 분석가야.
과장하지 말고 리뷰 데이터에 근거해 작성해.

다음은 수업용 가상 리뷰를 LLM이 1~5점으로 평가한 결과야.
평균 평점과 리뷰 내용을 바탕으로 마케팅/기획 담당자가 읽을 수 있는 짧은 종합 보고서를 작성해줘.

상품명: {product_name}
평균 평점: {average_rating}

리뷰 평가 데이터:
{rated_reviews}

출력 형식:
1. 종합 평가: 2~3문장
2. 긍정 포인트: bullet 3개
3. 개선 포인트: bullet 3개
4. 마케팅/기획 제안: bullet 3개
""")



In [ ]:
review_report_chain = review_report_prompt | review_llm | StrOutputParser()


In [ ]:
review_report = review_report_chain.invoke({
    "product_name": review_scored_df["product_name"].iloc[0],
    "average_rating": average_rating,
    "rated_reviews": rated_reviews_text,
})

print(review_report)


### 문제 2. 리뷰 평가 기준 바꿔보기

아래 중 하나를 바꿔 실행해 보세요.

- 평점 기준에서 `배송 문제는 -1점 반영` 같은 규칙 추가
- 보고서 출력 형식에 `타깃 고객군별 인사이트` 추가
- `data/synthetic_product_reviews.csv`에 리뷰 3개를 더 추가한 뒤 평균 평점 변화 확인

이 예시는 리뷰를 외부에 게시하기 위한 것이 아니라, **텍스트 리뷰 분석 자동화와 보고서 작성 흐름을 연습하기 위한 수업용 예시**입니다.

<details>
<summary>정답 보기</summary>

아래처럼 평점 기준에 배송 감점 규칙을 추가하고, 보고서 출력 형식에 타깃 고객군별 인사이트를 추가할 수 있습니다. 프롬프트를 바꾼 뒤에는 평점화 셀과 보고서 작성 셀을 다시 실행합니다.

```python
review_rating_prompt = ChatPromptTemplate.from_template("""
너는 이커머스 리뷰 분석가입니다.
리뷰 내용을 읽고 고객 만족도를 1~5점 정수로 평가하세요.
1점은 매우 불만족, 3점은 보통 또는 장단점 혼재, 5점은 매우 만족을 의미합니다.
배송 지연, 파손, 오배송 문제가 명확하면 최종 점수에서 1점을 낮춰 평가하세요.
단, 제품 만족도가 매우 높고 배송 문제가 사소하면 감점하지 않아도 됩니다.
반드시 JSON만 출력하세요.

상품명: {product_name}
고객군: {customer_segment}
리뷰: {review_text}

출력 형식:
{{"rating": 4, "reason": "평점 근거 한 문장", "key_issue": "핵심 이슈 한 단어 또는 짧은 구"}}
""")

review_rating_chain = review_rating_prompt | review_llm | review_rating_parser
```

```python
review_report_prompt = ChatPromptTemplate.from_template("""
너는 마케팅 리서치 보고서를 작성하는 분석가야.
과장하지 말고 리뷰 데이터에 근거해 작성해.

다음은 수업용 가상 리뷰를 LLM이 1~5점으로 평가한 결과야.
평균 평점과 리뷰 내용을 바탕으로 마케팅/기획 담당자가 읽을 수 있는 짧은 종합 보고서를 작성해줘.

상품명: {product_name}
평균 평점: {average_rating}

리뷰 평가 데이터:
{rated_reviews}

출력 형식:
1. 종합 평가: 2~3문장
2. 긍정 포인트: bullet 3개
3. 개선 포인트: bullet 3개
4. 타깃 고객군별 인사이트: bullet 3개
5. 마케팅/기획 제안: bullet 3개
""")
```
</details>



## 7. 카테고리 분류

카테고리 분류는 상담 문의, 뉴스 기사, VOC를 미리 정한 업무 분류로 나누는 작업입니다.  
프롬프트에 가능한 카테고리 목록을 넣으면, 분류 기준을 더 안정적으로 만들 수 있습니다.


In [ ]:
inquiry_df = pd.DataFrame({  # 카테고리 분류에 사용할 고객 문의 예시를 만듭니다.
    "text": [
        "주문한 상품이 아직 도착하지 않았습니다.",
        "불량 제품을 받아서 환불하고 싶습니다.",
        "쿠폰 적용이 안 되는데 확인 부탁드립니다.",
        "제품 사용 중 전원이 자꾸 꺼집니다.",
        "회원 탈퇴 메뉴가 어디 있는지 모르겠습니다.",
    ]
})

categories = ["배송", "환불/교환", "가격/쿠폰", "상품품질", "계정/기타"]  # 모델이 선택할 수 있는 카테고리 목록입니다.
inquiry_df  # 분류 대상 문의 목록을 확인합니다.


In [ ]:
category_prompt = ChatPromptTemplate.from_template("""  # 카테고리 분류용 프롬프트를 구성합니다.
너는 고객 문의를 업무 카테고리로 분류하는 분석가야.

다음 문의를 카테고리 중 하나로 JSON 분류해줘.
반드시 아래 형식의 JSON만 출력해줘.

출력 형식:
{{"category": "...", "reason": "..."}}

카테고리:
{categories}

문의:
{text}
""")

category_chain = category_prompt | llm | JsonOutputParser()  # 프롬프트 -> 모델 -> JSON 파서 체인을 만듭니다.

category_rows = []  # 문의별 분류 결과를 저장할 리스트입니다.
for text in inquiry_df["text"]:  # 문의를 한 건씩 분류합니다.
    category_rows.append(category_chain.invoke({
        "categories": ", ".join(categories),  # 카테고리 리스트를 쉼표로 이어진 문자열로 전달합니다.
        "text": text,  # 현재 분류할 문의 문장입니다.
    }))

category_df = pd.concat([inquiry_df, pd.DataFrame(category_rows)], axis=1)  # 원본 문의와 분류 결과를 합칩니다.
category_df  # 최종 카테고리 분류 결과를 확인합니다.



## 8. 쇼핑몰 문의 답변 자동화

이번에는 LLM을 사용해 쇼핑몰 고객 문의에 대한 답변 초안을 자동으로 만들어 봅니다.  
중요한 점은 LLM에게 마음대로 답하게 하는 것이 아니라, **쇼핑몰 정책을 함께 전달하고 그 정책 안에서만 답변하게 만드는 것**입니다.

이 예시는 실제 고객 데이터가 아니라 수업용으로 만든 **가상 문의 데이터**를 사용합니다. 실제 업무에서는 개인정보, 주문번호, 연락처 같은 민감한 정보가 모델 입력에 포함되지 않도록 반드시 마스킹하거나 제외해야 합니다.

흐름은 다음과 같습니다.

1. `data/shopping_mall_inquiries.csv`에서 가상 문의를 읽습니다.
2. 쇼핑몰 운영 정책이 담긴 텍스트 파일을 읽습니다.
3. 각 문의와 정책을 함께 LLM에 전달합니다.
4. 생성된 답변을 `answer` 컬럼에 저장합니다.
5. 결과를 새 CSV 파일로 저장합니다.


In [ ]:
support_df = pd.read_csv("data/shopping_mall_inquiries.csv")  # 수업용 가상 쇼핑몰 문의 데이터를 읽습니다.
support_df  # answer 컬럼은 아직 비어 있고, LLM 답변으로 채울 예정입니다.


In [ ]:
from pathlib import Path  # 텍스트 파일 경로를 다루기 위해 사용합니다.

SHOPPING_MALL_POLICY = Path("data/shopping_mall_policy.txt").read_text(encoding="utf-8")  # 쇼핑몰 정책 파일을 읽습니다.
print(SHOPPING_MALL_POLICY[:700])  # 정책 내용 앞부분을 확인합니다.


In [ ]:
support_llm = ChatOpenAI(model="gpt-5-nano")  # 비용이 낮은 모델로 고객 문의 답변 초안을 생성합니다.


In [ ]:
support_reply_prompt = ChatPromptTemplate.from_template("""
너는 온라인 쇼핑몰 고객센터 상담원입니다.
반드시 제공된 쇼핑몰 정책 안에서만 답변하세요.
정책에 없는 내용은 추측하지 말고 고객센터 확인이 필요하다고 안내하세요.
답변은 3~5문장으로 작성하고, 고객이 다음에 무엇을 하면 되는지 분명히 알려주세요.

쇼핑몰 정책:
{policy}

고객 문의:
{question}

정책 기반 고객 답변을 작성해줘.
""")

support_reply_chain = support_reply_prompt | support_llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 답변 체인입니다.



In [ ]:
answers = []
for question in support_df["question"]:  # 문의를 한 건씩 처리합니다.
    answer = support_reply_chain.invoke({
        "policy": SHOPPING_MALL_POLICY,
        "question": question,
    })
    answers.append(answer)

answers[:2]  # 생성된 답변 일부를 먼저 확인합니다.


In [ ]:
support_df["answer"] = answers  # 질문 오른쪽 answer 컬럼에 LLM 답변을 채웁니다.
support_df.to_csv("data/shopping_mall_inquiries_with_answers.csv", index=False, encoding="utf-8-sig")  # 답변이 채워진 결과를 저장합니다.

support_df[["inquiry_id", "category", "question", "answer"]]  # 최종 답변 결과를 확인합니다.


### 문제 3. 정책을 바꿔 답변 변화 확인하기

아래 중 하나를 바꿔 실행해 보세요.

- `SHOPPING_MALL_POLICY`에서 반품 가능 기간을 7일에서 14일로 변경
- `SHOPPING_MALL_POLICY`에 `예약상품은 출고일이 별도로 안내됩니다`라는 정책 추가
- `question` 하나를 직접 추가해 답변이 어떻게 달라지는지 확인

이 실습의 핵심은 LLM이 답변을 만들어내더라도, 답변의 기준은 **프롬프트에 넣은 정책**에서 나온다는 점입니다. 실제 업무에서는 생성된 답변을 바로 발송하지 말고 상담원이 검토한 뒤 사용하는 방식이 안전합니다.

<details>
<summary>정답 보기</summary>

```python
SHOPPING_MALL_POLICY_UPDATED = SHOPPING_MALL_POLICY.replace(
    "단순 변심 교환/반품은 상품 수령 후 7일 이내 신청할 수 있습니다.",
    (
        "단순 변심 교환/반품은 상품 수령 후 14일 이내 신청할 수 있습니다."
        "\n- 예약상품은 출고일이 별도로 안내됩니다."
    ),
)

test_question = "예약상품은 언제 배송되나요? 그리고 단순 변심 반품은 며칠 안에 신청할 수 있나요?"

answer = support_reply_chain.invoke({
    "policy": SHOPPING_MALL_POLICY_UPDATED,
    "question": test_question,
})

print(answer)
```
</details>



## 9. OpenAI 임베딩으로 문서 유사도 계산

임베딩은 텍스트를 모델이 계산할 수 있는 **실수형 벡터**로 바꾸는 과정입니다. 문장이나 문서가 숫자 배열로 바뀌면, 두 벡터의 거리를 계산해 의미가 비슷한 문서를 찾을 수 있습니다.

<img src="image/text_embedding_vector_flow.svg" width="820">

앞에서는 텍스트를 벡터로 바꾸고 벡터 간 유사도를 계산하는 개념을 살펴봤습니다.  
이번에는 `naver_economy_news.csv`에 저장된 뉴스 제목과 본문을 OpenAI 임베딩으로 바꾼 뒤, 검색 문장과 가장 비슷한 뉴스를 찾아봅니다.

이 방식은 검색, 추천, 중복 문서 탐지, RAG의 검색 단계에서 핵심적으로 사용됩니다.  
예를 들어 사용자가 `스타벅스 선불카드 환불 논란`처럼 질문하면, 전체 뉴스 중 의미가 가장 가까운 기사부터 정렬할 수 있습니다.



In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")  # 문장을 임베딩 벡터로 바꿀 모델입니다.


In [ ]:
news_embedding_df = pd.read_csv("naver_economy_news.csv")  # 수집해 둔 경제 뉴스 데이터를 읽습니다.
news_embedding_df = news_embedding_df.dropna(subset=["title"]).copy()  # 제목이 없는 기사는 검색 대상에서 제외합니다.
news_embedding_df["body"] = news_embedding_df["body"].fillna("").astype(str)  # 본문 결측치는 빈 문자열로 처리합니다.
news_embedding_df["press"] = news_embedding_df["press"].fillna("").astype(str)  # 언론사 결측치를 정리합니다.
news_embedding_df["time"] = news_embedding_df["time"].fillna("").astype(str)  # 시간 결측치를 정리합니다.
news_embedding_df["url"] = news_embedding_df["url"].fillna("").astype(str)  # URL 결측치를 정리합니다.

news_embedding_df[["title", "press", "time"]].head()


In [ ]:
article_limit = 20  # 임베딩할 기사 수입니다. API 비용과 실행 시간을 고려해 일부만 사용합니다.
body_char_limit = 600  # 기사 1건당 본문에서 사용할 글자 수입니다.

news_search_df = news_embedding_df.head(article_limit).copy()  # 검색 대상으로 사용할 기사 일부를 선택합니다.

# 제목과 본문을 함께 넣으면 제목만 넣을 때보다 문서 의미를 더 잘 반영할 수 있습니다.
news_search_df["embedding_text"] = news_search_df.apply(
    lambda row: (
        f"제목: {row['title']}\n"
        f"언론사: {row['press']}\n"
        f"본문 일부: {row['body'][:body_char_limit]}"
    ),
    axis=1,
)

text_docs = news_search_df["embedding_text"].tolist()  # 임베딩할 뉴스 문서 목록입니다.
print("검색 대상 기사 수:", len(text_docs))
print(text_docs[0][:500])


In [ ]:
query = "스타벅스 선불카드 환불 기준과 공정위 약관 논란"  # 검색 질문 또는 관심 주제입니다.
query


In [ ]:
doc_vectors = embedding_model.embed_documents(text_docs)  # 뉴스 문서들을 각각 임베딩 벡터로 변환합니다.
query_vector = embedding_model.embed_query(query)  # 검색 문장도 같은 임베딩 공간의 벡터로 변환합니다.

print("검색 대상 기사 수:", len(doc_vectors))  # 임베딩한 문서 개수를 확인합니다.
print("임베딩 차원:", len(query_vector))  # 임베딩 벡터의 길이를 확인합니다.
print("쿼리 벡터 앞 5개 값:", query_vector[:5])  # 벡터가 실제 숫자 배열인지 일부 값을 확인합니다.


In [ ]:
def cosine_similarity(vec_a, vec_b):  # 두 벡터의 코사인 유사도를 계산하는 함수입니다.
    a = np.array(vec_a)  # 첫 번째 벡터를 numpy 배열로 바꿉니다.
    b = np.array(vec_b)  # 두 번째 벡터를 numpy 배열로 바꿉니다.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))  # 내적을 벡터 크기로 나눠 유사도를 계산합니다.

similarities = [cosine_similarity(query_vector, doc_vector) for doc_vector in doc_vectors]  # 쿼리와 각 뉴스의 유사도를 계산합니다.

similarity_df = news_search_df[["title", "press", "time", "url"]].copy()  # 검색 결과에 보여줄 기사 정보를 가져옵니다.
similarity_df["query"] = query  # 어떤 검색 문장으로 찾았는지 함께 기록합니다.
similarity_df["similarity"] = similarities  # 계산된 유사도를 추가합니다.
similarity_df["body_preview"] = news_search_df["body"].str.slice(0, 160) + "..."  # 본문 일부만 미리보기로 보여줍니다.

similarity_df = similarity_df.sort_values("similarity", ascending=False).reset_index(drop=True)  # 유사도가 높은 기사부터 정렬합니다.
similarity_df[["similarity", "title", "press", "time", "body_preview", "url"]].head(5)  # 검색 결과 상위 5개를 확인합니다.


### 문제 4. 나만의 뉴스 임베딩 검색 만들기

아래 중 하나를 바꿔 실행해 보세요.

- `query`를 `고금리 대출을 낮은 금리로 갈아타는 지원 정책`으로 변경
- `query`를 `커피 브랜드 선불카드 환불과 소비자 보호 논란`으로 변경
- `article_limit`을 30으로 늘려 더 많은 기사에서 검색
- 유사도가 가장 높은 기사 3개만 출력

<details>
<summary>정답 보기</summary>

```python
article_limit = 30
body_char_limit = 600

news_search_df = news_embedding_df.head(article_limit).copy()
news_search_df["embedding_text"] = news_search_df.apply(
    lambda row: (
        f"제목: {row['title']} | "
        f"언론사: {row['press']} | "
        f"본문 일부: {row['body'][:body_char_limit]}"
    ),
    axis=1,
)

text_docs = news_search_df["embedding_text"].tolist()
query = "고금리 대출을 낮은 금리로 갈아타는 지원 정책"

doc_vectors = embedding_model.embed_documents(text_docs)
query_vector = embedding_model.embed_query(query)

similarities = [cosine_similarity(query_vector, doc_vector) for doc_vector in doc_vectors]

similarity_df = news_search_df[["title", "press", "time", "url"]].copy()
similarity_df["query"] = query
similarity_df["similarity"] = similarities
similarity_df["body_preview"] = news_search_df["body"].str.slice(0, 160) + "..."

similarity_df = similarity_df.sort_values("similarity", ascending=False).reset_index(drop=True)
similarity_df[["similarity", "title", "press", "url"]].head(3)
```
</details>



## 체크포인트

- 이전에는 텍스트를 수집·정제·토큰화하고, 벡터화와 분류 모델로 통계 기반 텍스트 분석을 구성했습니다.
- LDA는 정답 라벨 없이 BoW 기반 단어 빈도 행렬에서 잠재 주제를 찾는 비지도학습 방법이며, LLM 분석 전에 반복 이슈를 빠르게 훑는 기준점으로 사용할 수 있습니다.
- LLM API는 규칙이나 통계만으로 판단하기 어렵고 사람이 문맥을 읽어야 하는 작업을 1차로 정리하는 데 유용합니다.
- 요약, 키워드 추출, 감정 분류, 카테고리 분류처럼 판단 기준이 필요한 작업을 프롬프트로 자동화할 수 있습니다.
- 출력 형식을 JSON으로 고정하면 결과를 데이터프레임으로 정리하기 쉽습니다.
- 가상 리뷰 CSV를 LLM으로 평점화하고, 평균 평점과 종합 리뷰 보고서를 자동으로 만들 수 있습니다.
- LLM API에 쇼핑몰 정책을 함께 전달하면 가상 문의에 대한 정책 기반 고객 문의 답변 초안을 생성할 수 있습니다.
- OpenAI 임베딩은 문장을 의미 벡터로 바꾸며, 코사인 유사도로 문서 검색과 추천을 구성할 수 있습니다.
- LLM 결과는 항상 검토가 필요합니다. 중요한 업무에서는 샘플 검수, 기준 문서화, 재현성 확인을 함께 해야 합니다.

